# Example how to read modify and _publishable content using the Confluence API


## Imports and setup

In [ ]:
import yaml
import requests

with open('foryouandyourteam.yaml') as f:
    configuration = yaml.safe_load(f)

confluence = configuration['confluence']
assert len(confluence['apiurl']) > 0

authentication = (confluence['username'], confluence['password'])

(confluence['apiurl'], confluence['space'], confluence['page'])

In [ ]:
import xml.dom.minidom
import IPython

def beautify(flat_xml):
    dom = xml.dom.minidom.parseString('<root doc="this artificial entity encapsulaes the Confluence content">{}</root>'.format(flat_xml))
    pretty_xml_as_string = dom.toprettyxml()
    return pretty_xml_as_string

def analyze_http_respnse(response):
    if not response.status_code == requests.codes.ok:
        from pprint import pprint
        from urllib.error import HTTPError
        pprint(vars(response))
        raise HTTPError(url, response.status_code, response.json(), response.headers, None)

# Read the page content

`curl -X GET -vv -H "Content-Type: application/json"   https://xyz.projects.atlassian.net/wiki/rest/api/content --user bue@foryouandyourcustomers.com:***`

In [ ]:
resp = requests.get(confluence['apiurl']+'/rest/api/content', auth=authentication, params= {
               'title': confluence['page'],
               'spaceKey': confluence['space'],
               'expand': 'body.view,version'
            }
        )

analyze_http_respnse(resp)
resp.raise_for_status()
resp

# Dump API response

In [ ]:
response = resp.json().get('results')
response

## Display the current content with syntax higlighted HTML

In [ ]:
page = response[0]
current_content = page['body']['view']['value']
IPython.display.Code(beautify(current_content))

# Apply changes

In [ ]:
current_version = page['version']['number']
version = current_version + 1
change_message = 'Bla'.format(version)

content = current_content + '.' #current_content + '<h2>Version {}</h2><p>Modified from {}</p>'.format(version, 'bla') 

In [ ]:
IPython.display.Code(beautify(content))

Beware of the version field format

In [ ]:
import json

d = {
    'title': page['title'],
    'id': page['id'],
    'type': page['type'],
    'version': { 'number': version, 'notifyEdit': False, 'message': change_message },
    'body': {
        'storage': {
            'representation': 'storage',
            'value': content
        }
    }
}

json.dumps(d)

## Publish the updated content

*HTTPError: 409 Client Error* indicates, either version or content are invalid

In [ ]:
url = page['_links']['self']

put_response = requests.put(url, headers={'Content-Type': 'application/json'}, json=d, auth=authentication)
d['version']['number'] = int(d['version']['number']) + 1

analyze_http_respnse(put_response)

In [ ]:
put_response

# Attachments

In [ ]:
page_id = put_response.json()['id']
page_id

In [ ]:
resp = requests.get(f"{confluence['apiurl']}/rest/api/content/{page_id}/child/attachment", auth=authentication, params= {
               'filename': 'diagram.drawio'
            }
        )
resp

In [ ]:
attachment = resp.json()
attachment

In [ ]:
attachment_id = attachment['results'][0]['id']
json = {
    'type': 'attachment',
    'fileName': 'diagram-z.drawio',
    'contentType': 'application/vnd.jgraph.mxfile',
    'comment': 'Automated update',
    'minorEdit': 'true',
    'metadata' : { "labels": [ { "name": "drawio" } ] }
}
with open('diagram2.drawio', 'rb') as src:
    content = src.read()
    
files = {"file":('diagram-z.drawio', content, 'application/vnd.jgraph.mxfile')}

headers = {"X-Atlassian-Token": "no-check", "Accept": "application/json"}

#resp = requests.post(f"{confluence['apiurl']}/rest/api/content/{page_id}/child/attachment/{attachment_id}/data", 
#                    auth=authentication, data=json, files=files, headers=headers
#        )


In [ ]:
resp = requests.post(f"{confluence['apiurl']}/rest/api/content/{page_id}/child/attachment", 
                    auth=authentication, data=json, files=files, headers=headers)

In [ ]:
resp, resp.text

In [ ]:
resp.json()

# Macro

```
<ac:structured-macro ac:name="inc-drawio" ac:schema-version="1"
                                     ac:macro-id="822e430e-1205-4318-af50-964bbeb13198">
                    <ac:parameter ac:name="diagramName">diagram.drawio</ac:parameter>
                    <ac:parameter ac:name="aspect">-IuDeWdp_pBGzQphX35I 1</ac:parameter>
                    <ac:parameter ac:name="includedDiagram">1</ac:parameter>
                    <ac:parameter ac:name="width">1492</ac:parameter>
                    <ac:parameter ac:name="aspectHash">dbc7e8e42f15a7eeff70d61877cbd88529b35777</ac:parameter>
                    <ac:parameter ac:name="pageId">139403187</ac:parameter>
                    <ac:parameter ac:name=""/>
                </ac:structured-macro>
```
```
<ac:structured-macro ac:name="inc-drawio" ac:schema-version="1"
                                     ac:macro-id="4da462ae-8d6a-497c-884b-5aa826e8ec0f">
                    <ac:parameter ac:name="border">false</ac:parameter>
                    <ac:parameter ac:name="diagramName">diagram.drawio</ac:parameter>
                    <ac:parameter ac:name="aspect">-IuDeWdp_pBGzQphX35I 1</ac:parameter>
                    <ac:parameter ac:name="includedDiagram">1</ac:parameter>
                    <ac:parameter ac:name="simpleViewer">false</ac:parameter>
                    <ac:parameter ac:name="width">1200</ac:parameter>
                    <ac:parameter ac:name="aspectHash">dbc7e8e42f15a7eeff70d61877cbd88529b35777</ac:parameter>
                    <ac:parameter ac:name="links">blank</ac:parameter>
                    <ac:parameter ac:name="tbstyle">inline</ac:parameter>
                    <ac:parameter ac:name="pageId">139403187</ac:parameter>
                    <ac:parameter ac:name="diagramDisplayName">DiagramTitleHere</ac:parameter>
                    <ac:parameter ac:name="lbox">true</ac:parameter>
                    <ac:parameter ac:name=""/>
                </ac:structured-macro>
```


In [ ]:
"""
<ac:structured-macro ac:macro-id="2016ef77-3a67-46f7-803d-8a6d035d0811" ac:name="drawio" ac:schema-version="1">
    <ac:parameter ac:name="border">true</ac:parameter>
    <ac:parameter ac:name="diagramName">Sample</ac:parameter>
    <ac:parameter ac:name="simpleViewer">false</ac:parameter>
    <ac:parameter ac:name="width"/>
    <ac:parameter ac:name="links">auto</ac:parameter>
    <ac:parameter ac:name="tbstyle">top</ac:parameter>
    <ac:parameter ac:name="lbox">true</ac:parameter>
    <ac:parameter ac:name="diagramWidth">121</ac:parameter>
    <ac:parameter ac:name="revision">1</ac:parameter>
    <ac:parameter ac:name=""/>
  </ac:structured-macro>
"""